# Pump-Probe Analysis Example

This notebook reproduces the key steps of a time-resolved X-ray scattering
experiment: generating synthetic data, normalising to an intensity reference,
correcting slow drift, and applying a timetool jitter correction.

All data is synthetic — generated by `escape.storage.example_data` — so the
notebook runs without any real data files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import escape
from escape.storage.example_data import make_pump_probe_scan

print(f"escape version: {escape.__version__}")

## Synthetic Dataset

`make_pump_probe_scan` returns four Arrays sharing the same pulse-ID index:

| Array | Description |
|-------|-------------|
| `sig` | Detector signal (both pumped and unpumped shots) |
| `i0` | Incoming X-ray intensity (correlated with `sig`) |
| `pump_on` | Boolean per event: `True` = laser was fired |
| `delay` | Nominal delay value per event |

In [ ]:
sig, i0, pump_on, delay = make_pump_probe_scan(
    n_steps=20,
    n_events_per_step=600,
    noise=0.06,
    i0_noise=0.04,
    seed=42,
)

print(f"Total events: {len(sig.index)}")
print()
print(sig.scan.par_steps.head())

## Step 1 — Separate Pump-On and Pump-Off Shots

Boolean indexing with a pulse-ID-aligned Array preserves scan structure.

In [ ]:
sig_on  = sig[~pump_on]   # laser-pumped signal
sig_off = sig[pump_on]    # unpumped reference

i0_on  = i0[~pump_on]
i0_off = i0[pump_on]

print(f"ON  events per step (first 3): {sig_on.scan.count()[:3]}")
print(f"OFF events per step (first 3): {sig_off.scan.count()[:3]}")

## Step 2 — Normalise to I0

The `/` operator finds common pulse IDs automatically — no explicit alignment needed.

In [ ]:
sig_on_norm  = sig_on  / i0_on
sig_off_norm = sig_off / i0_off

print(f"Normalised ON  shape: {sig_on_norm.shape}")
print(f"Normalised OFF shape: {sig_off_norm.shape}")

## Step 3 — Per-Step Pump/Probe Ratio

Compute the relative signal change ΔI/I per scan step.

In [ ]:
means_on  = np.asarray(sig_on_norm.scan.nanmean())
means_off = np.asarray(sig_off_norm.scan.nanmean())

ratio = means_on / means_off - 1.0   # relative change ΔI/I

par_vals  = np.asarray(sig.scan.par_steps["delay_s"])
delays_ps = par_vals * 1e12

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delays_ps, ratio, "o-", lw=1.5)
ax.axhline(0, ls="--", color="k", alpha=0.4)
ax.set_xlabel("delay / ps")
ax.set_ylabel("ΔI/I")
ax.set_title("Pump-probe signal (no jitter correction)")
plt.tight_layout()
plt.show()

## Step 4 — Timetool Jitter Correction with `digitize`

Timing jitter between pump and probe broadens the apparent response.
We correct it by re-binning events onto the timetool-measured actual delay
using `escape.digitize`.

Here we simulate a timetool measurement by adding Gaussian jitter (~100 fs rms)
to the nominal delay.

In [ ]:
rng = np.random.default_rng(0)

# True delay = nominal + random jitter per event
jitter = rng.normal(0, 100e-15, len(sig.index))
t_actual = escape.Array(
    data=(np.repeat(par_vals, 600) + jitter).astype(np.float64),
    index=sig.index,
    step_lengths=sig.scan.step_lengths,
    parameter=sig.scan.parameter,
    name="t_actual_s",
)

# Re-bin onto 50 fs bins covering the delay range
t_bins  = np.arange(par_vals.min(), par_vals.max(), 50e-15)
t_sorted = escape.digitize(t_actual.compute(), t_bins)

# Apply the bin ordering to the normalised ON signal
ratio_tt = t_sorted.categorize(sig_on_norm)

print(f"Number of time bins: {len(ratio_tt.scan)}")

In [ ]:
ratio_tt_mean = np.asarray(ratio_tt.scan.nanmean())
ratio_tt_sem  = np.asarray(ratio_tt.scan.nanstd()) / np.sqrt(
    np.asarray(ratio_tt.scan.count(), dtype=float)
)
t_centers_ps  = np.asarray(ratio_tt.scan.par_steps.iloc[:, 0]) * 1e12

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(t_centers_ps, ratio_tt_mean / ratio_tt_mean.mean() - 1,
            yerr=ratio_tt_sem, fmt="o-", lw=1.5, label="timetool corrected")
ax.plot(delays_ps, ratio, "s--", alpha=0.5, label="nominal delay")
ax.axhline(0, ls="--", color="k", alpha=0.3)
ax.set_xlabel("delay / ps")
ax.set_ylabel("ΔI/I")
ax.set_title("Pump-probe signal — with and without jitter correction")
ax.legend()
plt.tight_layout()
plt.show()

## Step 5 — Storing Results

`Array.store()` persists data and full scan metadata into HDF5.
The file can be reloaded in a later session with `Array.load_from_h5()`.

In [ ]:
with h5py.File("pump_probe_results.h5", "w") as fh:
    sig.store(fh,            "signal")
    i0.store(fh,             "i0")
    pump_on.store(fh,        "pump_on")
    sig_on_norm.store(fh,    "signal_norm_on")
    ratio_tt.store(fh,       "signal_tt_corrected")

print("Saved. Contents:")
with h5py.File("pump_probe_results.h5", "r") as fh:
    fh.visit(print)